# Notebook 12: Inheritance

Inheritance lets a class acquire the properties and behaviors of another class, enabling code reuse and modeling **'is-a'** relationships.

When you write `class Dog : public Animal`, you are saying that every `Dog` **is an** `Animal` — it has all of Animal's data and methods, and can add its own on top.

## Is-a vs Has-a

There are two fundamental ways to reuse code between classes:

| Relationship | Mechanism | Example |
|---|---|---|
| **is-a** | Inheritance | A `Dog` is-a `Animal` |
| **has-a** | Composition | A `Car` has-a `Engine` |

**Inheritance (is-a):** The derived class *is a kind of* the base class. A `Dog` can do everything an `Animal` can, plus more.

**Composition (has-a):** A class *contains* an instance of another class as a member. A `Car` has an `Engine` object inside it.

> **Rule of thumb:** Prefer **has-a** (composition) over **is-a** (inheritance) when in doubt. Inheritance creates tight coupling between classes. Use inheritance only when the relationship is truly a specialisation.

## Basic Inheritance Syntax

```cpp
class Derived : public Base {
    // additional members
};
```

Let's build a simple shape hierarchy.

In [ ]:
#include <iostream>
#include <string>

class Shape {
public:
    std::string name;

    Shape(const std::string &n) : name(n) {}

    double area() const {
        return 0.0;
    }

    void printName() const {
        std::cout << "Shape: " << name << std::endl;
    }
};

In [ ]:
class Rectangle : public Shape {
public:
    double width;
    double height;

    Rectangle(double w, double h)
        : Shape("Rectangle"), width(w), height(h) {}

    double area() const {
        return width * height;
    }
};

// Instantiation and method calls
Rectangle rect(4.0, 3.0);
rect.printName();                          // inherited from Shape
std::cout << "Area: " << rect.area() << std::endl;  // Rectangle's own area()

## What Gets Inherited

With `public` inheritance:

- **public** members of the base → accessible everywhere (inherited as public)
- **protected** members of the base → accessible inside derived class only
- **private** members of the base → exist in memory but **cannot be accessed directly** from derived class

The derived class contains all the data of the base — private members are physically present, you just cannot name them directly.

In [ ]:
#include <iostream>
#include <string>

// rect was defined in the previous cell — state persists
std::cout << "Name from derived: " << rect.name << std::endl;  // public member of Shape
std::cout << "Width: " << rect.width << std::endl;              // Rectangle's own member
std::cout << "Area via inherited method: " << rect.area() << std::endl;

## Constructor Chaining

A derived class **must** initialise the base class part of its object. This is done in the **initialiser list**:

```cpp
Derived(args) : Base(base_args), ownMember(val) {}
```

If you do not call the base constructor explicitly, C++ calls the **default (no-argument) constructor** of the base. If the base has no default constructor, you get a compile error.

In [ ]:
#include <iostream>
#include <string>

class Animal {
public:
    std::string species;
    int legs;

    Animal(const std::string &s, int l) : species(s), legs(l) {
        std::cout << "Animal constructor called for " << species << std::endl;
    }
};

class Dog : public Animal {
public:
    std::string breed;

    // Explicitly chain to Animal constructor
    Dog(const std::string &b) : Animal("Canis lupus familiaris", 4), breed(b) {
        std::cout << "Dog constructor called for breed: " << breed << std::endl;
    }
};

Dog d("Labrador");
std::cout << "Species: " << d.species << ", Legs: " << d.legs << ", Breed: " << d.breed << std::endl;

**Exercise 1:** Create a base class `Vehicle` with a `std::string brand`, an `int year`, a constructor that initialises both, and a `printInfo()` method that prints them. Create a derived class `Truck` with an `int payload` (in tons). Give `Truck` its own constructor that chains to `Vehicle`'s constructor.

In [ ]:
// Your code here

## Access Specifiers in Inheritance

The access specifier after the colon controls how inherited members are visible to the outside world:

| Base member | `public` inheritance | `protected` inheritance | `private` inheritance |
|---|---|---|---|
| `public` | public | protected | private |
| `protected` | protected | protected | private |
| `private` | inaccessible | inaccessible | inaccessible |

- **`public` inheritance** — models **is-a**. Most common. Use this by default.
- **`private` inheritance** — models **implemented-in-terms-of**. All inherited members become private. Rarely needed; usually composition is better.
- **`protected` inheritance** — very rare. Public members become protected.

> For virtually all OOP code, use `public` inheritance.

## Protected Members

`protected` members are accessible:
- Inside the class itself
- Inside any derived class
- **Not** from outside code

Use `protected` when you want derived classes to be able to use a helper but do not want it exposed as part of the public API.

In [ ]:
#include <iostream>
#include <string>

class Logger {
protected:
    void log(const std::string &msg) const {
        std::cout << "[LOG] " << msg << std::endl;
    }

public:
    virtual void doWork() {}
};

class FileProcessor : public Logger {
public:
    void process(const std::string &filename) {
        log("Starting processing of " + filename);  // OK: derived class using protected
        // ... processing ...
        log("Done processing " + filename);
    }
};

FileProcessor fp;
fp.process("data.txt");
// fp.log("direct");  // ERROR: protected is not accessible from outside

## Method Overriding (Non-Virtual)

A derived class can define a method with the same name as one in the base class. Without `virtual`, this is **static dispatch** — the method chosen at compile time based on the **pointer/reference type**, not the actual object type.

This can produce surprising results:

In [ ]:
#include <iostream>

// rect and Rectangle are already defined above — using them here

Shape *s = new Rectangle(5.0, 2.0);
std::cout << "Via Shape pointer: area = " << s->area() << std::endl;
// Prints 0 — calls Shape::area(), NOT Rectangle::area()!
// The compiler looks at the pointer type (Shape*) to decide which method to call.

Rectangle *r = new Rectangle(5.0, 2.0);
std::cout << "Via Rectangle pointer: area = " << r->area() << std::endl;
// Prints 10 — calls Rectangle::area() because pointer type is Rectangle*

delete s;
delete r;

// This is the problem that virtual functions solve (next notebook).

**Exercise 2:** Create a class `Animal2` (avoid redefining `Animal` from above) with a `speak()` method that prints `"Some animal sound"`. Create a class `Dog2` derived from `Animal2` with its own `speak()` that prints `"Woof"`. Create a `Dog2` object and call `speak()` directly on it. Then assign it to an `Animal2` pointer and call `speak()` through that pointer. Observe the difference and explain it in a comment.

In [ ]:
// Your code here

## Multiple Levels of Inheritance

Inheritance can be chained: `C` inherits from `B`, which inherits from `A`. `C` has all members of both `A` and `B`.

> Keep hierarchies **shallow** in practice — 2 to 3 levels maximum. Deep hierarchies become hard to reason about.

In [ ]:
#include <iostream>
#include <string>

class LivingThing {
public:
    bool isAlive;
    LivingThing() : isAlive(true) {
        std::cout << "LivingThing constructed" << std::endl;
    }
};

class Vertebrate : public LivingThing {
public:
    int spineCount;
    Vertebrate(int s) : spineCount(s) {
        std::cout << "Vertebrate constructed" << std::endl;
    }
};

class Cat : public Vertebrate {
public:
    std::string name;
    Cat(const std::string &n) : Vertebrate(33), name(n) {
        std::cout << "Cat constructed: " << name << std::endl;
    }
};

Cat c("Whiskers");
std::cout << "Alive: " << c.isAlive         // from LivingThing
          << ", Spine: " << c.spineCount    // from Vertebrate
          << ", Name: " << c.name << std::endl;  // from Cat

## Inheritance and the Orthodox Canonical Form

When you destroy a derived object, destructors run in **reverse order**: derived destructor first, then base destructor automatically.

**Important warning:** If you delete a derived object through a **base pointer** and the base destructor is **not virtual**, the derived destructor is **not called** — undefined behaviour. We will fix this in the polymorphism notebook with `virtual` destructors.

In [ ]:
#include <iostream>
#include <string>

class BaseOCF {
public:
    BaseOCF()  { std::cout << "BaseOCF constructor" << std::endl; }
    ~BaseOCF() { std::cout << "BaseOCF destructor" << std::endl; }
};

class DerivedOCF : public BaseOCF {
public:
    DerivedOCF()  { std::cout << "DerivedOCF constructor" << std::endl; }
    ~DerivedOCF() { std::cout << "DerivedOCF destructor" << std::endl; }
};

{
    DerivedOCF obj;  // Constructors: Base first, then Derived
    std::cout << "--- object in use ---" << std::endl;
}  // Destructors: Derived first, then Base

std::cout << std::endl << "--- Danger: deleting via base pointer (non-virtual destructor) ---" << std::endl;
BaseOCF *ptr = new DerivedOCF();
delete ptr;  // Only calls ~BaseOCF() — ~DerivedOCF() is SKIPPED! Undefined behaviour.

## Final Exercise

Design a small employee hierarchy:

1. `Employee` — has `std::string name` and `double salary`. Constructor takes both. Has a method `describe()` that prints `"Employee: [name], salary: [salary]"`.
2. `Manager` — derives from `Employee`. Adds `std::string department`. Constructor chains to `Employee`. Overrides `describe()` to also print the department.
3. `Intern` — derives from `Employee`. Adds `std::string university`. Constructor chains to `Employee`. Overrides `describe()` to also print the university.

Create one instance of each and call `describe()` directly on each object (not through pointers yet).

In [ ]:
// Your code here

## Modern C++ (C++11 and Beyond)

C++11 introduced several quality-of-life improvements for inheritance.

### `override` keyword
Placed after the method signature in the derived class. The compiler verifies that the method actually overrides a virtual method in the base — it errors if you have a typo or wrong signature.

### `final` keyword
- On a class: `class Concrete final` — no class can inherit from it.
- On a method: `virtual void foo() final` — no further override allowed.

### Inherited constructors
`using Base::Base;` brings all base constructors into the derived class — you do not need to write forwarding constructors.

In [ ]:
#include <iostream>
#include <string>

class BaseModern {
public:
    std::string label;
    BaseModern(const std::string &l) : label(l) {}
    virtual void describe() const {
        std::cout << "Base: " << label << std::endl;
    }
    virtual ~BaseModern() {}
};

class DerivedModern : public BaseModern {
public:
    using BaseModern::BaseModern;  // C++11: inherit all constructors from Base

    void describe() const override {  // C++11: compiler checks this really overrides
        std::cout << "Derived: " << label << std::endl;
    }
};

// Prevents anyone inheriting from Sealed
class Sealed final : public BaseModern {
public:
    using BaseModern::BaseModern;
    void describe() const override final {
        std::cout << "Sealed: " << label << std::endl;
    }
};

DerivedModern dm("hello");  // uses inherited constructor
dm.describe();

Sealed s("locked");
s.describe();